# 05 - Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

### Adding New Features/Columns

In [3]:
df.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice',
       'Latitude', 'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'SubdivisionName', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet'],
      dtype='object')

In [4]:
import geopandas as gpd
from utils import create_time_split, get_preprocessing_pipeline


school_districts = gpd.read_file("california_school_districts.geojson")  
unified_districts = school_districts[school_districts["DistrictType"] == "Unified"].copy()

df = pd.read_csv('Data/cleaned_sold.csv')

properties_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs="EPSG:4326"
)

if properties_gdf.crs != unified_districts.crs:
    properties_gdf = properties_gdf.to_crs(unified_districts.crs)

properties_enriched = gpd.sjoin(
    properties_gdf, 
    unified_districts[['DistrictName', 'geometry']], 
    how="left", 
    predicate="within"  
)

df = properties_enriched.drop(columns=['geometry', 'index_right'])


df['CloseDate'] = pd.to_datetime(df['CloseDate'])

df['BedBathRatio'] = df['BedroomsTotal'] / np.maximum(df['BathroomsTotalInteger'], 1)
df['AgeProperty'] =  (df['CloseDate'].dt.year - df['YearBuilt']).clip(lower=0)

df['BedBathRatio'] = df['BedBathRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
df['AgeProperty'] = df['AgeProperty'].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df.drop(columns = ["Flooring"]) #too many nulls and weird formatting, had to remove for better performance/less errors

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

def extract_date_features(dataframe, date_col):
    df_feat = dataframe.copy()
    df_feat[f'{date_col}_year'] = df_feat[date_col].dt.year
    df_feat[f'{date_col}_month'] = df_feat[date_col].dt.month
    df_feat[f'{date_col}_day'] = df_feat[date_col].dt.day
    df_feat[f'{date_col}_dayofweek'] = df_feat[date_col].dt.dayofweek
    df_feat = df_feat.drop(columns=[date_col])
    return df_feat

X_train_raw = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test_raw = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

X_train_numeric = extract_date_features(X_train_raw, 'CloseDate')
X_test_numeric = extract_date_features(X_test_raw, 'CloseDate')

preprocessor = get_preprocessing_pipeline(X_train_numeric)
X_train_processed = preprocessor.fit_transform(X_train_numeric)
X_test_processed = preprocessor.transform(X_test_numeric)

Training Window (X=12 months): 2025-05-30 to 2026-05-30 | Rows: 122208
Testing Window (1 month): 2026-05-30 to 2026-06-30


### Retraining Baseline Linear Model

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

model = LinearRegression()
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

MSE: 138411149388.54865
R² Score: 0.8251668962669066
MAPE: 0.2154550984169947
MdAPE: 0.14652978951266765


### Retraining Decision Tree Model

In [6]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import GridSearchCV

decision_tree = DecisionTreeRegressor(random_state=42)

param_grid = {
    'criterion': ['squared_error'],
    'max_depth': [None, 3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

grid_search = GridSearchCV(
    estimator=decision_tree,
    param_grid=param_grid,
    cv=3,                            
    scoring='r2', 
    n_jobs=-1,                       
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score (Negative MSE):", grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

Fitting 3 folds for each of 135 candidates, totalling 405 fits
Best Hyperparameters: {'criterion': 'squared_error', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best Cross-Validation Score (Negative MSE): 0.7311181909015553
MSE: 195690336208.97388
R² Score: 0.7528150802798113
MAPE: 0.17482682113853099
MdAPE: 0.10869565217391304


### Retraining Random Forest Model

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import RandomizedSearchCV

rf_model = RandomForestRegressor()

rf_param_distributions = {
    'n_estimators': [50, 100],
    'criterion': ['squared_error'],
    'max_depth': [5, 10, 20, None],
    'min_samples_split':[2, 5, 10, 15],
    'min_samples_leaf':[1, 2, 4, 8],
    'max_features': ['sqrt', 'log2']
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_distributions,
    n_iter=10,                       
    cv=3,                            
    scoring='r2',                     
    n_jobs=-1,                       
    random_state=42,
    verbose=1,
    error_score='raise'               
)

rf_random_search.fit(X_train_processed, y_train)

print("Best RF Hyperparameters:", rf_random_search.best_params_)
print("Best RF Cross-Validation R² Score:", rf_random_search.best_score_)

best_rf_model = rf_random_search.best_estimator_
y_pred = best_rf_model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best RF Hyperparameters: {'n_estimators': 50, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'criterion': 'squared_error'}
Best RF Cross-Validation R² Score: 0.8317854784279755
MSE: 127998459581.4395
R² Score: 0.8383196147092364
MAPE: 0.15741981126314974
MdAPE: 0.09933606702569449


### Comparing Model Performance

In [8]:
compare_table = pd.DataFrame({
    "Model": [
        "Linear Regression", 
        "Decision Tree",
        "Random Forest"
    ],
    "Test R2 Before": [
        0.8408, 
        0.7631,
        0.8065  
    ],
    "Test R2 After": [
        0.8396, 
        0.7358,
        0.8316
    ]
})

compare_table

,Model,Test R2 Before,Test R2 After
0,Linear Regression,0.8408,0.8396
1,Decision Tree,0.7631,0.7358
2,Random Forest,0.8065,0.8316
